---
## Section 0 — Configuration (Structured Streaming)

**Prérequis :** Session 2 §2.8 (Parquet consolidé) et ideally Session 3 (Delta).

Si vous ouvrez **Session 4 seule** (kernel vide), exécutez **cette cellule** avant les autres pour créer `spark`, les chemins (`STREAM_SOURCE_DIR`, `DELTA_*`, …) et les dossiers de streaming.


In [16]:
import os
import sys
import platform
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

# Mac Apple Silicon : Java arm64 + workers PySpark arm64 (identique Session 3)
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _jdk_candidates = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
    ]
    _jdk = next((p for p in _jdk_candidates if (p / "bin" / "java").exists()), None)
    if _jdk:
        os.environ["JAVA_HOME"] = str(_jdk)
        os.environ["PATH"] = f"{_jdk / 'bin'}:{os.environ.get('PATH', '')}"
    _python = Path(sys.executable)
    _wrapper = _python.parent / "python-arm64"
    if not _wrapper.exists():
        _wrapper.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{_python}" "$@"\n',
            encoding="utf-8",
        )
        _wrapper.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(_wrapper)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(_python)

# Chemins projet
_candidats = [Path("data"), Path("../data")]
DATA_DIR = next((p for p in _candidats if (p / "velib").exists()), _candidats[0])
OUTPUT_DIR = DATA_DIR / "output"
VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR = OUTPUT_DIR / "stream_input"
STREAM_CHECKPOINT = OUTPUT_DIR / "checkpoints"

assert VELIB_CONSOLIDE.exists(), (
    f"Fichier manquant : {VELIB_CONSOLIDE.resolve()}\n"
    "Exécutez Spark_DIA3_Session_2.ipynb §2.8 avant ce notebook."
)

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

APP_NAME = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8


def _clear_spark_session_registry() -> None:
    from pyspark.sql.context import SQLContext

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None


def _spark_is_alive() -> bool:
    if "spark" not in globals() or spark is None:
        return False
    try:
        _sc = spark.sparkContext
        return _sc is not None and _sc._jsc is not None
    except Exception:
        return False


def _delta_is_configured() -> bool:
    if not _spark_is_alive():
        return False
    try:
        extensions = spark.conf.get("spark.sql.extensions", "")
        catalog = spark.conf.get("spark.sql.catalog.spark_catalog", "")
        return (
            "DeltaSparkSessionExtension" in extensions
            and "DeltaCatalog" in catalog
        )
    except Exception:
        return False


def reset_spark_session() -> None:
    global spark, sc

    if _spark_is_alive():
        try:
            spark.stop()
        except Exception as exc:
            print(f"spark.stop() ignoré : {exc}")

    if "sc" in globals() and sc is not None:
        try:
            if sc._jsc is not None:
                sc.stop()
        except Exception as exc:
            print(f"sc.stop() ignoré : {exc}")

    _clear_spark_session_registry()
    spark = None
    sc = None


def create_delta_spark_session():
    """Session Spark avec Delta Lake (obligatoire pour .format('delta'))."""
    global spark, sc

    reset_spark_session()

    _builder = (
        SparkSession.builder
        .appName(APP_NAME)
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(_builder).getOrCreate()
    sc = spark.sparkContext
    sc.setLogLevel("WARN")

    if not _delta_is_configured():
        raise RuntimeError(
            "Delta Lake non actif. Kernel → Restart, puis relancez cette cellule."
        )

    print(f"Spark {spark.version} — Delta Lake activé")
    return spark, sc


def stop_streaming_query(name: str) -> bool:
    """Arrête une requête streaming nommée (utile en réexécution de cellule)."""
    for q in spark.streams.active:
        if q.name == name:
            q.stop()
            return True
    return False


if not _spark_is_alive() or not _delta_is_configured():
    create_delta_spark_session()
else:
    sc = spark.sparkContext
    print(f"Spark déjà actif ({spark.version}) — Delta Lake OK")

print(f"[OK] stream_input : {STREAM_SOURCE_DIR.resolve()}")


Spark déjà actif (3.5.9) — Delta Lake OK
[OK] stream_input : /Users/romain/Desktop/SparkVelib/data/output/stream_input


---
# PARTIE 2 -- Structured Streaming (après-midi)

## 2.1 Concepts fondamentaux

### Du batch au streaming

Le batch traite un ensemble de données **figé** : on sait à l'avance combien
il y a de lignes, on peut trier, on peut faire des jointures complètes.

Le streaming traite un flux de données **continu** : les données arrivent
en permanence, on ne connaît pas la fin, et certains événements peuvent
arriver en retard.

Spark Structured Streaming répond à ce problème avec un modèle élégant :
le flux est traité comme une **table infinie** qui s'agrandit au fil du temps.
Les requêtes SQL ou DataFrame s'y appliquent sans modification.

```
Flux d'entrée  :  [evt1] [evt2] [evt3] ...  [evtN]  ...
                     |      |      |              |
                  +--+------+------+--------------+----→ temps
                  |
                  Spark : exécution de micro-batchs toutes les X secondes
                  |
                  Résultat : table de résultats mise à jour en continu
```

### Vocabulaire

| Terme | Définition |
|-------|-----------|
| **Trigger** | Fréquence d'exécution des micro-batchs |
| **Checkpoint** | Sauvegarde de l'état pour la reprise après panne |
| **Watermark** | Délai maximal toléré pour les données tardives |
| **Fenêtre glissante** | Agrégation sur une plage de temps mobile |
| **Sink** | Destination de l'écriture (console, fichier, Delta...) |


---
## 2.2 Le simulateur de flux

En production, le flux proviendrait de Kafka ou de l'API GBFS en direct.
Pour ce cours, un script Python **rejoue les données historiques** en écrivant
des fichiers JSON dans un répertoire surveillé par Spark.

Ce mécanisme -- la **file source** (file source) -- est le moyen le plus simple
de tester Structured Streaming sans infrastructure externe.

> Le simulateur est fourni dans `scripts/simulateur_flux.py`.
> Lancez-le dans un terminal séparé **avant** d'exécuter les cellules suivantes.
>
> ```bash
> python scripts/simulateur_flux.py --output data/output/stream_input --vitesse 3
> ```
>
> L'option `--vitesse 3` signifie que chaque seconde réelle correspond à
> 3 minutes de données historiques.


In [17]:
# Vérification : le simulateur tourne-t-il ?
import time
from pathlib import Path

def compter_fichiers_stream(attente_sec: int = 5) -> list[Path]:
    """Attend et retourne les fichiers JSON produits par le simulateur."""
    time.sleep(attente_sec)
    return sorted(
        Path(STREAM_SOURCE_DIR).glob("*.json"),
        key=lambda p: p.stat().st_mtime,
    )

fichiers = compter_fichiers_stream(3)
n = len(fichiers)
if n == 0:
    print("[ATTENTION] Aucun fichier JSON trouvé dans le répertoire de streaming.")
    print("  Vérifiez que le simulateur tourne dans un terminal séparé.")
    print(f"  Répertoire surveillé : {STREAM_SOURCE_DIR}")
else:
    dernier = fichiers[-1]
    print(f"[OK] {n} fichier(s) JSON détecté(s).")
    print(f"  Dernier fichier : {dernier.name}")
    print("  Aperçu :")
    print(dernier.read_text(encoding="utf-8")[:400])


[OK] 161 fichier(s) JSON détecté(s).
  Dernier fichier : snapshot_000075.json
  Aperçu :
[{"station_id": 653099465, "nom_station": "Amédée Huon - Fort d'Ivry", "code_arr": 653099, "capacite": 31, "velos_meca": 1, "velos_elec": 1, "bornettes_libres": 29, "horodatage": "2020-11-26T18:47Z"}, {"station_id": 1810975577, "nom_station": "Guy Môquet - Parvis Georges Marchais", "code_arr": 1810975, "capacite": 30, "velos_meca": 0, "velos_elec": 1, "bornettes_libres": 29, "horodatage": "2020-11


---
## 2.3 Source de streaming : `readStream`

`spark.readStream` fonctionne exactement comme `spark.read`, avec deux différences :

1. Il retourne un **DataFrame de streaming** (pas un DataFrame ordinaire).
2. Il ne peut pas être affiché directement avec `.show()` -- il faut déclencher
   une **requête de streaming** avec `.writeStream`.


In [20]:
# Schéma du flux JSON produit par le simulateur
from pathlib import Path
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

if "STREAM_SOURCE_DIR" not in globals():
    _candidats = [Path("data"), Path("../data")]
    _data = next((p for p in _candidats if (p / "output").exists()), _candidats[0])
    STREAM_SOURCE_DIR = _data / "output" / "stream_input"

schema_flux = StructType([
    StructField("station_id",       IntegerType(),   False),
    StructField("nom_station",      StringType(),    True),
    StructField("code_arr",         IntegerType(),   True),
    StructField("capacite",         IntegerType(),   True),
    StructField("velos_meca",       IntegerType(),   True),
    StructField("velos_elec",       IntegerType(),   True),
    StructField("bornettes_libres", IntegerType(),   True),
    StructField("horodatage",       StringType(),    False),
])

# Création du DataFrame de streaming — au plus 2 fichiers par micro-batch
stream_df = (
    spark.readStream
    .schema(schema_flux)
    .option("maxFilesPerTrigger", 2)
    .json(str(STREAM_SOURCE_DIR))
)

print(f"Est un streaming DataFrame : {stream_df.isStreaming}")
print(f"Colonnes : {stream_df.columns}")
# On ne peut pas appeler .count() ou .show() sur un streaming DataFrame


Est un streaming DataFrame : True
Colonnes : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage']


In [21]:
# Ajout des colonnes calculées — exactement comme en batch
from pyspark.sql.functions import col, greatest, lit, round as spark_round, to_timestamp, when

stream_enrichi = (
    stream_df
    .withColumn("horodatage", to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mm'Z'"))
    .filter(col("capacite") > 0)
    .withColumn("velos_meca", greatest(col("velos_meca"), lit(0)))
    .withColumn("velos_elec", greatest(col("velos_elec"), lit(0)))
    .withColumn("bornettes_libres", greatest(col("bornettes_libres"), lit(0)))
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
    .withColumn(
        "taux_occupation",
        spark_round((col("capacite") - col("bornettes_libres")) / col("capacite"), 4),
    )
    .withColumn(
        "taux_occupation",
        when(col("taux_occupation") < 0, 0.0)
        .when(col("taux_occupation") > 1, 1.0)
        .otherwise(col("taux_occupation")),
    )
    .withColumn("est_vide", col("taux_occupation") < 0.10)
)

print("Colonnes après enrichissement :", stream_enrichi.columns)


Colonnes après enrichissement : ['station_id', 'nom_station', 'code_arr', 'capacite', 'velos_meca', 'velos_elec', 'bornettes_libres', 'horodatage', 'velos_total', 'taux_occupation', 'est_vide']


---
## 2.4 Première requête : sink console

La sortie `console` écrit les résultats dans le terminal Jupyter.
C'est utile pour le débogage -- jamais pour la production.


In [22]:
# Requête de streaming vers la console
import time
from pathlib import Path
from pyspark.sql.functions import col, greatest, lit, round as spark_round, when

# Enrichissement (même logique que la cellule précédente)
stream_console = (
    stream_df
    .filter(col("capacite") > 0)
    .withColumn("velos_meca", greatest(col("velos_meca"), lit(0)))
    .withColumn("velos_elec", greatest(col("velos_elec"), lit(0)))
    .withColumn("bornettes_libres", greatest(col("bornettes_libres"), lit(0)))
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
    .withColumn(
        "taux_occupation",
        spark_round((col("capacite") - col("bornettes_libres")) / col("capacite"), 4),
    )
    .withColumn(
        "taux_occupation",
        when(col("taux_occupation") < 0, 0.0)
        .when(col("taux_occupation") > 1, 1.0)
        .otherwise(col("taux_occupation")),
    )
    .withColumn("est_vide", col("taux_occupation") < 0.10)
    .select(
        "station_id",
        "nom_station",
        "code_arr",
        "horodatage",
        "velos_total",
        "taux_occupation",
        "est_vide",
    )
)

_checkpoint_console = Path(STREAM_CHECKPOINT) / "console"
_checkpoint_console.mkdir(parents=True, exist_ok=True)

q_console = (
    stream_console
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", str(_checkpoint_console))
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(20)
print(f"Statut de la requête : {q_console.status}")
print(
    f"Lignes lues (dernier batch) : "
    f"{q_console.lastProgress['numInputRows'] if q_console.lastProgress else 'N/A'}"
)
q_console.stop()
print("Requête console arrêtée.")


26/07/22 09:47:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 25
-------------------------------------------
+----------+---------------------------------+--------+-----------------+-----------+---------------+--------+
|station_id|nom_station                      |code_arr|horodatage       |velos_total|taux_occupation|est_vide|
+----------+---------------------------------+--------+-----------------+-----------+---------------+--------+
|1034247348|Place des Larris                 |1034247 |2020-11-26T19:47Z|2          |0.0588         |true    |
|653035822 |Jeûneurs - Mulhouse              |653035  |2020-11-26T19:47Z|2          |0.0714         |true    |
|27414460  |Danielle Casanova - Place Vendôme|27414   |2020-11-26T19:47Z|3          |0.0811         |true    |
|60667075  |Georges V - François 1er         |60667   |2020-11-26T19:47Z|1          |0.0455         |true    |
|52453     |Le Vau - Maurice Bertaux         |52      |2020-11-26T19:47Z|2          |0.08           |true    |
|66491368  |So

26/07/22 09:47:50 ERROR TorrentBroadcast: Store broadcast broadcast_33 fail, remove all pieces of the broadcast


In [23]:
# Équivalent Python pur (sans PySpark) — micro-batchs toutes les 5 s, max 2 fichiers
import json
import time
from pathlib import Path

stream_dir = Path(STREAM_SOURCE_DIR)

def enrichir_ligne(ligne: dict) -> dict | None:
    capacite = ligne.get("capacite")
    if not capacite or capacite <= 0:
        return None

    velos_meca = max(ligne.get("velos_meca") or 0, 0)
    velos_elec = max(ligne.get("velos_elec") or 0, 0)
    bornettes_libres = max(ligne.get("bornettes_libres") or 0, 0)
    velos_total = velos_meca + velos_elec

    taux = round((capacite - bornettes_libres) / capacite, 4)
    taux = max(0.0, min(1.0, taux))

    return {
        "station_id": ligne.get("station_id"),
        "nom_station": ligne.get("nom_station"),
        "code_arr": ligne.get("code_arr"),
        "horodatage": ligne.get("horodatage"),
        "velos_total": velos_total,
        "taux_occupation": taux,
        "est_vide": taux < 0.10,
    }

def lire_fichier_json(chemin: Path) -> list[dict]:
    donnees = json.loads(chemin.read_text(encoding="utf-8"))
    return donnees if isinstance(donnees, list) else [donnees]

def afficher_batch(numero: int, lignes: list[dict], max_lignes: int = 10) -> None:
    print("-" * 43)
    print(f"Batch: {numero}")
    print("-" * 43)
    if not lignes:
        print("(aucune ligne)")
        return
    cols = [
        "station_id", "nom_station", "code_arr", "horodatage",
        "velos_total", "taux_occupation", "est_vide",
    ]
    print(" | ".join(cols))
    for ligne in lignes[:max_lignes]:
        print(" | ".join(str(ligne[c]) for c in cols))
    if len(lignes) > max_lignes:
        print(f"only showing top {max_lignes} rows")

deja_vus: set[Path] = set()
batch_num = 0
fin = time.time() + 20

while time.time() < fin:
    time.sleep(5)

    fichiers = sorted(stream_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
    nouveaux = [f for f in fichiers if f not in deja_vus][:2]

    lignes_enrichies: list[dict] = []
    for chemin in nouveaux:
        deja_vus.add(chemin)
        for ligne in lire_fichier_json(chemin):
            enrichie = enrichir_ligne(ligne)
            if enrichie:
                lignes_enrichies.append(enrichie)

    afficher_batch(batch_num, lignes_enrichies)
    batch_num += 1

print(f"Micro-batchs affichés : {batch_num}")
print("Simulation console Python arrêtée.")


-------------------------------------------
Batch: 0
-------------------------------------------
station_id | nom_station | code_arr | horodatage | velos_total | taux_occupation | est_vide
54000559 | Jouffroy d'Abbans - Wagram | 54000 | 2020-11-26T19:05Z | 2 | 0.05 | True
207633746 | Grande Armée - Brunel | 207633 | 2020-11-26T19:05Z | 4 | 0.0645 | True
210566542 | Square Boucicaut | 210566 | 2020-11-26T19:05Z | 3 | 0.05 | True
213682736 | Morillons - Dantzig | 213682 | 2020-11-26T19:05Z | 5 | 0.0962 | True
653208733 | Romainville - Vaillant-Couturier | 653208 | 2020-11-26T19:05Z | 3 | 0.0789 | True
129094535 | Porte de Saint-Ouen - Bessières | 129094 | 2020-11-26T19:05Z | 4 | 0.0952 | True
54000612 | Cambrai - Benjamin Constant | 54000 | 2020-11-26T19:05Z | 4 | 0.093 | True
653132534 | Louis Lejeune - Barbès | 653132 | 2020-11-26T19:05Z | 2 | 0.08 | True
506455295 | Général Martial Valin - Pont du Garigliano | 506455 | 2020-11-26T19:05Z | 1 | 0.0625 | True
210408693 | Vaugirard - Mons

---
## 2.5 Agrégations sur fenêtres temporelles glissantes

L'agrégation sur des **fenêtres temporelles** est la requête streaming la plus
courante en pratique. Elle répond à des questions du type :
"Combien de vélos sont disponibles par arrondissement sur les 10 dernières minutes ?"

Spark propose deux types de fenêtres temporelles :

| Type | Définition | Exemple |
|------|-----------|---------|
| Fenêtre **basculante** | Fenêtres fixes, sans chevauchement | 0-10 min, 10-20 min... |
| Fenêtre **glissante** | Fenêtres qui se chevauchent | [0-10], [5-15], [10-20]... |

```
Fenêtre basculante (10 min)  : |--w1--|--w2--|--w3--|
Fenêtre glissante (10/5)   : |--w1--|          Taille=10, pas=5
                                  |--w2--|
                                       |--w3--|
```


In [24]:
from pyspark.sql.functions import avg, col, count, greatest, lit, to_timestamp, window

# Enrichissement intégré (évite un stream_enrichi obsolète en mémoire)
stream_pour_fenetres = (
    stream_df
    .withColumn("horodatage", to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mm'Z'"))
    .filter(col("capacite") > 0)
    .withColumn("velos_meca", greatest(col("velos_meca"), lit(0)))
    .withColumn("velos_elec", greatest(col("velos_elec"), lit(0)))
    .withColumn("velos_total", col("velos_meca") + col("velos_elec"))
)

# Agrégation glissante : disponibilité moyenne par arrondissement
# Fenêtre de 10 minutes, avançant toutes les 2 minutes
df_fenetre_arr = (
    stream_pour_fenetres
    .withWatermark("horodatage", "5 minutes")
    .groupBy(
        col("code_arr"),
        window(col("horodatage"), "10 minutes", "2 minutes"),
    )
    .agg(
        avg("velos_total").alias("velos_moyens"),
        count("*").alias("nb_mesures"),
    )
    .select(
        col("code_arr"),
        col("window.start").alias("fenetre_debut"),
        col("window.end").alias("fenetre_fin"),
        col("velos_moyens"),
        col("nb_mesures"),
    )
)

print("Schéma de la requête fenêtrée :")
df_fenetre_arr.printSchema()


Schéma de la requête fenêtrée :
root
 |-- code_arr: integer (nullable = true)
 |-- fenetre_debut: timestamp (nullable = true)
 |-- fenetre_fin: timestamp (nullable = true)
 |-- velos_moyens: double (nullable = true)
 |-- nb_mesures: long (nullable = false)



In [25]:
# Écriture vers Delta Lake (sink delta) -- mode append
import time
from pathlib import Path

if not _delta_is_configured():
    raise RuntimeError(
        "Delta Lake requis pour ce sink. Relancez la Section 0, puis readStream et fenêtres."
    )

path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
_checkpoint_fenetres = Path(STREAM_CHECKPOINT) / "fenetres_arr"
_checkpoint_fenetres.mkdir(parents=True, exist_ok=True)

if stop_streaming_query("fenetres_arrondissement"):
    print("Requête précédente 'fenetres_arrondissement' arrêtée — redémarrage.")

q_fenetres = (
    df_fenetre_arr
    .writeStream
    .outputMode("append")
    .format("delta")
    .option("checkpointLocation", str(_checkpoint_fenetres))
    .trigger(processingTime="10 seconds")
    .queryName("fenetres_arrondissement")
    .start(path_fenetres)
)

print(f"Requête démarrée : {q_fenetres.name}")
print(f"ID               : {q_fenetres.id}")
print(f"Sink Delta       : {path_fenetres}")
print("En attente de 3 batchs...")
time.sleep(35)
print(f"Dernier progrès : {q_fenetres.lastProgress}")


Requête précédente 'fenetres_arrondissement' arrêtée — redémarrage.
Requête démarrée : fenetres_arrondissement
ID               : 961aaf45-73d4-412a-b18f-60fedb7b7aaf
Sink Delta       : data/output/delta/fenetres_arrondissement
En attente de 3 batchs...


26/07/22 09:48:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Dernier progrès : {'id': '961aaf45-73d4-412a-b18f-60fedb7b7aaf', 'runId': 'c91a4ac0-2675-42c3-a866-79ef6128ce0a', 'name': 'fenetres_arrondissement', 'timestamp': '2026-07-22T07:48:40.005Z', 'batchId': 81, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 12, 'triggerExecution': 12}, 'eventTime': {'watermark': '2020-11-27T04:17:00.000Z'}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[file:/Users/romain/Desktop/SparkVelib/data/output/stream_input]', 'startOffset': {'logOffset': 80}, 'endOffset': {'logOffset': 80}, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'description': 'DeltaSink[data/output/delta/fenetres_arrondissement]', 'numOutputRows': -1}}


In [26]:
# Lecture des résultats accumulés dans Delta
path_fenetres = str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")

try:
    df_fenetres = spark.read.format("delta").load(path_fenetres)
    df_fenetres.orderBy("fenetre_debut").show(30, truncate=False)
except Exception as e:
    print(f"Pas encore de données : {e}")
    print("Attendez encore quelques secondes et relancez cette cellule.")


+--------+-------------------+-------------------+------------+----------+
|code_arr|fenetre_debut      |fenetre_fin        |velos_moyens|nb_mesures|
+--------+-------------------+-------------------+------------+----------+
|214122  |2020-11-26 18:56:00|2020-11-26 19:06:00|2.0         |1         |
|15409   |2020-11-26 18:56:00|2020-11-26 19:06:00|1.0         |1         |
|653035  |2020-11-26 18:56:00|2020-11-26 19:06:00|2.0         |1         |
|213686  |2020-11-26 18:56:00|2020-11-26 19:06:00|2.0         |1         |
|1810975 |2020-11-26 18:56:00|2020-11-26 19:06:00|2.0         |1         |
|102749  |2020-11-26 18:56:00|2020-11-26 19:06:00|2.0         |1         |
|208795  |2020-11-26 18:56:00|2020-11-26 19:06:00|1.0         |1         |
|129060  |2020-11-26 18:56:00|2020-11-26 19:06:00|1.0         |1         |
|581480  |2020-11-26 18:56:00|2020-11-26 19:06:00|5.0         |1         |
|66505   |2020-11-26 18:56:00|2020-11-26 19:06:00|4.0         |1         |
|102307  |2020-11-26 18:5

---
## 2.6 Alertes : détection de stations en rupture prolongée

L'objectif opérationnel est d'alerter l'équipe de maintenance quand une station
reste vide (zéro vélo disponible) pendant au moins **deux fenêtres consécutives**,
soit 20 minutes de rupture continue.

### Approche : `foreachBatch`

Pour une logique d'alerte complexe (état entre batchs, seuil consécutif),
`foreachBatch` est plus adapté que les agrégations pures. Il permet d'exécuter
une fonction Python arbitraire sur chaque micro-batch.


In [27]:
# Table d'alertes : schéma
from pyspark.sql.types import TimestampType

schema_alertes = StructType([
    StructField("station_id",   IntegerType(),   False),
    StructField("nom_station",  StringType(),    True),
    StructField("code_arr",     IntegerType(),   True),
    StructField("debut_rupture",TimestampType(), True),
    StructField("fin_rupture",  TimestampType(), True),
    StructField("duree_min",    IntegerType(),   True),
    StructField("ts_alerte",    TimestampType(), True),
])

# Initialisation de la table Delta d'alertes (vide)
(
    spark.createDataFrame([], schema_alertes)
    .write
    .format("delta")
    .mode("overwrite")
    .save(str(DELTA_ALERTES))
)
print(f"Table d'alertes initialisée : {DELTA_ALERTES}")


Table d'alertes initialisée : data/output/delta/alertes


In [28]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, count as spark_count, max as spark_max, min as spark_min, sum as spark_sum

SEUIL_RUPTURE_MIN = 20  # deux fenêtres consécutives de 10 minutes

# Mémoire inter-batchs : stations actuellement en rupture et depuis quand
etat_ruptures: dict[int, dict] = {}

def traiter_batch_alertes(batch_df, batch_id: int) -> None:
    """Fonction appelée par foreachBatch pour chaque micro-batch.

    Détecte les stations qui passent en rupture ou sortent de rupture.
    Écrit les alertes complètes dans la table Delta.

    Args:
        batch_df : DataFrame du micro-batch courant.
        batch_id : Identifiant séquentiel du batch.
    """
    global etat_ruptures

    if batch_df.isEmpty():
        return

    etat_batch = (
        batch_df
        .groupBy("station_id", "nom_station", "code_arr")
        .agg(
            spark_min("horodatage").alias("horodatage_min"),
            spark_max("horodatage").alias("horodatage_max"),
            spark_sum(col("est_vide").cast("int")).alias("nb_vide"),
            spark_count("*").alias("nb_obs"),
        )
        .collect()
    )

    alertes = []
    for row in etat_batch:
        sid = int(row["station_id"])
        en_rupture_batch = row["nb_vide"] == row["nb_obs"] and row["nb_obs"] > 0

        if en_rupture_batch and sid not in etat_ruptures:
            etat_ruptures[sid] = {
                "station_id": sid,
                "nom_station": row["nom_station"],
                "code_arr": row["code_arr"],
                "debut_rupture": row["horodatage_min"],
                "alerte_emise": False,
            }

        elif en_rupture_batch and sid in etat_ruptures:
            debut = etat_ruptures[sid]["debut_rupture"]
            fin = row["horodatage_max"]
            duree_min = int((fin - debut).total_seconds() // 60)
            if duree_min >= SEUIL_RUPTURE_MIN and not etat_ruptures[sid]["alerte_emise"]:
                alertes.append({
                    "station_id": sid,
                    "nom_station": etat_ruptures[sid]["nom_station"],
                    "code_arr": etat_ruptures[sid]["code_arr"],
                    "debut_rupture": debut,
                    "fin_rupture": fin,
                    "duree_min": duree_min,
                    "ts_alerte": fin,
                })
                etat_ruptures[sid]["alerte_emise"] = True

        elif not en_rupture_batch and sid in etat_ruptures:
            debut = etat_ruptures[sid]["debut_rupture"]
            fin = row["horodatage_max"]
            duree_min = int((fin - debut).total_seconds() // 60)
            if duree_min >= SEUIL_RUPTURE_MIN:
                alertes.append({
                    "station_id": sid,
                    "nom_station": etat_ruptures[sid]["nom_station"],
                    "code_arr": etat_ruptures[sid]["code_arr"],
                    "debut_rupture": debut,
                    "fin_rupture": fin,
                    "duree_min": duree_min,
                    "ts_alerte": fin,
                })
            del etat_ruptures[sid]

    if alertes:
        (
            spark.createDataFrame(alertes, schema=schema_alertes)
            .write
            .format("delta")
            .mode("append")
            .save(str(DELTA_ALERTES))
        )
        print(f"[batch {batch_id}] {len(alertes)} alerte(s) enregistrée(s)")

In [29]:
# Lancement de la requête d'alerte
import time
from pathlib import Path

_checkpoint_alertes = Path(STREAM_CHECKPOINT) / "alertes"
_checkpoint_alertes.mkdir(parents=True, exist_ok=True)

if stop_streaming_query("q_alertes_rupture"):
    print("Requête précédente 'q_alertes_rupture' arrêtée — redémarrage.")

q_alertes = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .writeStream
    .foreachBatch(traiter_batch_alertes)
    .outputMode("append")
    .option("checkpointLocation", str(_checkpoint_alertes))
    .trigger(processingTime="10 seconds")
    .queryName("q_alertes_rupture")
    .start()
)

print("Requête d'alertes démarrée. En attente de données...")
time.sleep(60)   # laisser tourner 1 minute pour accumuler des événements

# Lecture des alertes produites
df_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
print(f"\nAlertes enregistrées : {df_alertes.count()}")
df_alertes.orderBy(F.desc("ts_alerte")).show(20, truncate=False)

26/07/22 09:49:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Requête d'alertes démarrée. En attente de données...


26/07/22 09:49:17 ERROR MicroBatchExecution: Query q_alertes_rupture [id = 53df2b1b-fa85-4b12-929f-45ecae0e1cb0, runId = 1c4a1c44-079c-4040-9164-fa4cd29eed27] terminated with error
py4j.Py4JException: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/Users/romain/Desktop/SparkVelib/.venv-spark/lib/python3.12/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/romain/Desktop/SparkVelib/.venv-spark/lib/python3.12/site-packages/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/Users/romain/Desktop/SparkVelib/.venv-spark/lib/python3.12/site-packages/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/var/folders/lm/5bczwl3112g0f4dm_2_hzv0c0000gn/T/ipykernel_96299/3496802230.py", line 38, in traiter_batch_alertes
    s


Alertes enregistrées : 0
+----------+-----------+--------+-------------+-----------+---------+---------+
|station_id|nom_station|code_arr|debut_rupture|fin_rupture|duree_min|ts_alerte|
+----------+-----------+--------+-------------+-----------+---------+---------+
+----------+-----------+--------+-------------+-----------+---------+---------+



---
## 2.7 Données tardives et watermark

Dans un système distribué, certains événements arrivent avec du retard
(réseau saturé, capteur hors ligne temporairement...). Spark doit décider
combien de temps attendre avant de considérer une fenêtre comme fermée.

C'est le rôle du **watermark** : il définit le retard maximal toléré.

```
Watermark de 5 minutes :
  Si le timestamp max observé est 10h30, Spark considère que
  toutes les données avant 10h25 sont arrivées.
  Les fenêtres fermées avant 10h25 ne seront plus mises à jour.
```

### Illustration avec des données tardives simulées


In [30]:
import json
import datetime
import time

LOG_STREAMING = OUTPUT_DIR / "streaming_session4.log"

# Injection de données tardives :
# Écrire dans le répertoire de streaming un fichier avec des horodatages "dans le passé"
# Les dates sont dans le format ISO
# Le fichier est injecté dans le répertoire surveillé par Spark.
# Chaque ligne est homogène à une observation (ou snapshot).
# N.B. Ajouter une ligne dans le fichier de logs
def injecter_donnees_tardives(retard_minutes: int, nb_lignes: int = 20):
    """Écrit un fichier JSON avec des horodatages retardés.

    Args:
        retard_minutes : Retard à simuler (en minutes).
        nb_lignes      : Nombre de snapshots à générer.
    """
    fichiers = sorted(STREAM_SOURCE_DIR.glob("snapshot_*.json"))
    if not fichiers:
        print("Aucun fichier source — lancez d'abord le simulateur.")
        return

    with fichiers[-1].open(encoding="utf-8") as f:
        lignes = json.load(f)[:nb_lignes]

    dernier_ts = datetime.datetime.fromisoformat(
        str(lignes[0]["horodatage"]).replace("Z", "+00:00")
    )
    ts_retarde = (dernier_ts - datetime.timedelta(minutes=retard_minutes)).strftime(
        "%Y-%m-%dT%H:%MZ"
    )

    for ligne in lignes:
        ligne["horodatage"] = ts_retarde

    nom_fichier = f"tardif_{retard_minutes}min_{int(time.time())}.json"
    chemin = STREAM_SOURCE_DIR / nom_fichier
    with chemin.open("w", encoding="utf-8") as f:
        json.dump(lignes, f, ensure_ascii=False)

    with LOG_STREAMING.open("a", encoding="utf-8") as f:
        f.write(
            f"{datetime.datetime.now().isoformat()} | injection tardive "
            f"{retard_minutes} min | {nom_fichier} | {len(lignes)} lignes\n"
        )

    print(f"Injecté {len(lignes)} lignes (retard {retard_minutes} min) -> {nom_fichier}")

# Scénario 1 : retard de 3 min -- dans le watermark (5 min) -> données prises en compte
injecter_donnees_tardives(retard_minutes=3, nb_lignes=10)
time.sleep(15)

# Scénario 2 : retard de 12 min -- hors watermark -> données ignorées par les agrégations
injecter_donnees_tardives(retard_minutes=12, nb_lignes=10)
time.sleep(15)

print("\nStatut des requêtes actives :")
for q in spark.streams.active:
    prog = q.lastProgress
    print(f"  {q.name:<30} -- inputRows : {prog['numInputRows'] if prog else 'N/A'}")

Injecté 10 lignes (retard 3 min) -> tardif_3min_1784706634.json


26/07/22 09:50:40 WARN HDFSBackedStateStoreProvider: The state for version 81 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/07/22 09:50:40 WARN HDFSBackedStateStoreProvider: The state for version 81 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/07/22 09:50:40 WARN HDFSBackedStateStoreProvider: The state for version 81 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/07/22 09:50:40 WARN HDFSBackedStateStoreProvider: The state for version 81 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/07/22 09:50:40 WARN HDFSBackedStateStoreProvider: The state for version 81 doesn't exist in loadedMaps. Reading s

Injecté 10 lignes (retard 12 min) -> tardif_12min_1784706649.json

Statut des requêtes actives :
  fenetres_arrondissement        -- inputRows : 10


In [31]:
# [EXERCICE]
# Créez une nouvelle requête de streaming qui calcule,
# pour chaque arrondissement et sur une fenêtre basculante de 15 minutes,
# le MAXIMUM du taux_occupation observé.
# Écrivez le résultat en mode "update" vers la console.
#
# Rappel : fenêtre basculante = window(col, "15 minutes")  (sans 3e argument)
# ──────────────────────────────────────────────────────────────────────────

from pathlib import Path
from pyspark.sql.functions import col, max as spark_max, window

_checkpoint_exercice = Path(STREAM_CHECKPOINT) / "exercice"
_checkpoint_exercice.mkdir(parents=True, exist_ok=True)

if stop_streaming_query("exercice_taux_max_arr"):
    print("Requête précédente 'exercice_taux_max_arr' arrêtée — redémarrage.")

df_exercice = (
    stream_enrichi
    .withWatermark("horodatage", "5 minutes")
    .groupBy(
        col("code_arr"),
        window(col("horodatage"), "15 minutes"),
    )
    .agg(spark_max("taux_occupation").alias("taux_max"))
    .select(
        col("code_arr"),
        col("window.start").alias("fenetre_debut"),
        col("window.end").alias("fenetre_fin"),
        col("taux_max"),
    )
)

q_exercice = (
    df_exercice
    .writeStream
    .outputMode("update")
    .format("console")
    .option("checkpointLocation", str(_checkpoint_exercice))
    .trigger(processingTime="10 seconds")
    .queryName("exercice_taux_max_arr")
    .start()
)

print(f"Requête exercice démarrée : {q_exercice.name}")

Requête exercice démarrée : exercice_taux_max_arr


26/07/22 09:51:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-------------------+-------------------+--------+
|code_arr|      fenetre_debut|        fenetre_fin|taux_max|
+--------+-------------------+-------------------+--------+
|   54000|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0938|
|    NULL|2020-11-26 19:00:00|2020-11-26 19:15:00|    0.08|
|  213992|2020-11-26 19:00:00|2020-11-26 19:15:00|    0.02|
|  213691|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0476|
|  244400|2020-11-26 19:00:00|2020-11-26 19:15:00|   0.087|
|   84966|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0769|
|   82568|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0345|
|  129081|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0476|
|   13372|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0303|
|  392576|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0417|
|   13242|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0645|
| 1057179|2020-11-26 19:00:00|2020-11-26 19:15:00|  0.0333|
|  

---
## 2.8 Arrêt propre des requêtes et bilan

En production, les requêtes de streaming tournent indéfiniment.
Dans un contexte de cours, il faut les arrêter proprement pour libérer les ressources.


In [32]:
# Arrêt de toutes les requêtes actives
import datetime

LOG_STREAMING = OUTPUT_DIR / "streaming_session4.log"

actives = spark.streams.active
print(f"Requêtes actives à arrêter : {[q.name for q in actives]}")

for q in actives:
    q.stop()
    with LOG_STREAMING.open("a", encoding="utf-8") as f:
        f.write(
            f"{datetime.datetime.now().isoformat()} | arrêt requête | "
            f"{q.name} | id={q.id}\n"
        )
    print(f"  Arrêtée : {q.name}")

print("\nToutes les requêtes de streaming sont arrêtées.")

Requêtes actives à arrêter : ['fenetres_arrondissement', 'exercice_taux_max_arr']
  Arrêtée : fenetres_arrondissement
  Arrêtée : exercice_taux_max_arr

Toutes les requêtes de streaming sont arrêtées.


In [33]:
# Lecture finale des résultats accumulés
print("=== Résultats finaux ===")

print("\n-- Fenêtres arrondissement --")
try:
    df_fin_fenetres = spark.read.format("delta").load(
        str(DELTA_DISPONIBLE.parent / "fenetres_arrondissement")
    )
    print(f"  {df_fin_fenetres.count()} fenêtres enregistrées")
    df_fin_fenetres.orderBy(F.desc("fenetre_debut")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucun résultat : {e}")

print("\n-- Alertes rupture --")
try:
    df_fin_alertes = spark.read.format("delta").load(str(DELTA_ALERTES))
    print(f"  {df_fin_alertes.count()} alertes enregistrées")
    df_fin_alertes.orderBy(F.desc("ts_alerte")).show(10, truncate=False)
except Exception as e:
    print(f"  Aucune alerte : {e}")


=== Résultats finaux ===

-- Fenêtres arrondissement --
  22293 fenêtres enregistrées
+--------+-------------------+-------------------+------------+----------+
|code_arr|fenetre_debut      |fenetre_fin        |velos_moyens|nb_mesures|
+--------+-------------------+-------------------+------------+----------+
|586539  |2020-11-27 05:06:00|2020-11-27 05:16:00|1.0         |1         |
|653063  |2020-11-27 05:06:00|2020-11-27 05:16:00|1.0         |1         |
|1810975 |2020-11-27 05:06:00|2020-11-27 05:16:00|1.0         |1         |
|213691  |2020-11-27 05:06:00|2020-11-27 05:16:00|1.0         |1         |
|120564  |2020-11-27 05:06:00|2020-11-27 05:16:00|5.0         |1         |
|1765749 |2020-11-27 05:06:00|2020-11-27 05:16:00|1.0         |1         |
|89553   |2020-11-27 05:06:00|2020-11-27 05:16:00|2.0         |1         |
|256124  |2020-11-27 05:06:00|2020-11-27 05:16:00|3.0         |1         |
|128887  |2020-11-27 05:06:00|2020-11-27 05:16:00|1.0         |1         |
|213686  |2020

---
## Bilan du Jour 2

### Ce que nous avons fait

| Étape | Module | Concept clé |
|-------|--------|-------------|
| Vues temporaires et SQL de base | Spark SQL | `createOrReplaceTempView`, SQL natif |
| Questions métier analytiques | Spark SQL | `LEFT ANTI JOIN`, `CASE WHEN`, sous-requêtes |
| Fenêtrage analytique | Spark SQL / DataFrame | `OVER`, `LAG`, `LEAD`, `ROW_NUMBER`, `AVG OVER` |
| Écriture Delta Lake | Delta | `format("delta")`, partitionnement |
| Time-travel | Delta | `versionAsOf`, historique des opérations |
| Mise à jour incrémentale | Delta | `MERGE INTO`, `whenMatchedUpdateAll` |
| Source de streaming | Structured Streaming | `readStream`, schéma explicite, file source |
| Sink console | Structured Streaming | débogage, `outputMode("append")` |
| Fenêtres glissantes | Structured Streaming | `window()`, basculante vs glissante |
| Sink Delta | Structured Streaming | écriture transactionnelle en flux |
| Alertes avec `foreachBatch` | Structured Streaming | logique inter-batchs, état partagé |
| Watermark et late data | Structured Streaming | tolérance, fermeture de fenêtres |

### Points d'attention

- En `outputMode("append")`, Spark n'écrit que des lignes **nouvelles et définitives**.
  Pour les agrégations fenêtrées, cela implique un watermark : Spark attend que la fenêtre
  soit fermée avant d'émettre son résultat.
- `foreachBatch` est puissant mais introduit un état mutable (`etat_ruptures`) qui
  n'est **pas persisté dans le checkpoint**. En cas de redémarrage, l'état est perdu.
  Pour un système de production, il faudrait utiliser `mapGroupsWithState` ou
  stocker l'état dans Delta Lake.
- Le checkpoint est **obligatoire** pour toute requête qui ne doit pas repartir de zéro
  après un redémarrage.

### Pour demain (Jour 3)

La table Delta `disponibilite` (batch) et les résultats des fenêtres glissantes
(streaming) serviront de données d'entrée pour le Jour 3 : construction des features,
entraînement d'un modèle de régression avec MLlib, clustering des stations avec K-Means
et suivi des expériences avec MLflow.


In [ ]:
#spark.stop()
print("SparkSession arrêtée. À demain !")
